# Web scraping 💻

El siguiente taller nos permitirá tomar algunos datos para analizarlos y generar un asistente virtual que recomiende propiedades.


La página que se utilizará será *Ciencuadras.com* (https://www.ciencuadras.com/)  🤑 // *codeshare* (https://codeshare.io/) ⌨️


### 1. Librerias 📚

Las siguientes librerías nos permiten realizar la búsqueda de los datos de forma más eficiente y llevar a cabo los siguientes análisis.

In [24]:
import time # Para esperar que cargue la página y nos ayuda para evitar un bloqueo de la IP //Restricciones de la misma.

from selenium import webdriver #Para abrir la página web y navegar por ella.

from selenium.webdriver.common.by import By #Para buscar los elementos en la página web.

from selenium.webdriver.chrome.service import Service #Para abrir el navegador.

from selenium.webdriver.chrome.options import Options #Para abrir el navegador sin interfaz gráfica.

import pandas as pd #Para guardar los datos en un archivo .csv



### 2. Revisamos la página en la que se hará la búsqueda 🔍

Cien Cuadras > Clic derecho > Inspeccionar > HTML de la página, información, etc.

Buscar en la pagina la información se puede copiar de dos formas:

##### - *XPath*

Nos permite realizar la busqueda por el árbol de HTML.



In [25]:
'''<div class="card">
    <h1>Nombre del producto</h1>
    <span class="precio">120.000</span>
</div>'''

'<div class="card">\n    <h1>Nombre del producto</h1>\n    <span class="precio">120.000</span>\n</div>'

##### - *XPath complete*

Xpath Completo será la ruta donde esta la información.

In [26]:
'''/html/body/div[1]/div[2]/div[3]/h1'''

'/html/body/div[1]/div[2]/div[3]/h1'

### 3. Configuracion del navegador 🖥️


In [ ]:
def setup_driver():

    """Configura y retorna el driver de Selenium"""
    
    chrome_options = Options()  # Crea una instancia de opciones para Chrome

    chrome_options.add_argument("--headless")  # Modo sin interfaz gráfica

    chrome_options.add_argument("--disable-gpu")  # Desactiva el uso de GPU (innecesario en modo headless)

    chrome_options.add_argument("--no-sandbox")  # Mejora compatibilidad en Linux

    chrome_options.add_argument("--disable-dev-shm-usage")  # Evita errores de memoria compartida

    # Especifica la ruta a tu chromedriver (ajusta según tu configuración)

    driver = webdriver.Chrome(options=chrome_options) #Esto hará que se habrá el controlador de selenium en el navegador Chrome.
    return driver

### 4. Etiquetas de la página 🗂️

Vamos a realizar la búsqueda por cada card, y se llamará con el siguiente nombre:

⭐ a[_ngcontent-result-c245]

En este caso, se utilizó el selector CSS para encontrar todos los elementos "< a >" que tienen el atributo relacionado con la clase de "card". El selector CSS se emplea para seleccionar elementos HTML en función de sus atributos, clases, ID, etc.

Para tomar información más puntual (tipo de propiedad, precio, ubicación, etc.), se utilizará el comando "find_element", el cual permite extraer elementos específicos del HTML.


***elements = driver.find_elements(By.CSS_SELECTOR, "a[_ngcontent-result-c245]")***



Algunas páginas tienen la información identificada mediante atributos como `ID`, `CLASS_NAME`, o se puede acceder mediante un `CSS_SELECTOR`. Aquí algunos ejemplos prácticos:

---

#### ⭐ ID

Se utiliza cuando el elemento tiene un identificador único:

```python
fruits = driver.find_element(By.ID, "fruits")


#### ⭐ CLASS_NAME

Se usa cuando queremos acceder a un elemento por su clase:

```python
fruit = fruits.find_element(By.CLASS_NAME, "tomatoes")


#### ⭐ CSS_SELECTOR

Cuando los elementos no tienen ID ni CLASS_NAME, se puede utilizar un selector CSS personalizado:


```python
driver.find_element(By.CSS_SELECTOR, 'h1[_ngcontent-detail-c331]').text



---

In [ ]:

def get_links(driver, url):

    """Obtiene todos los enlaces de elementos con la clase especificada"""
    
    driver.get(url)
    time.sleep(3) # Espera a que la página cargue

    elements = driver.find_elements(By.CSS_SELECTOR, "a[_ngcontent-result-c245]") 

    links = [element.get_attribute('href') for element in elements if element.get_attribute('href')]
    
    return links


### 5. Extraer cada información de cada propiedad 🏠

En esta función se tomará datos de cada propiedad, que serán:

* Url
* Tipo de propiedad 
* Precio
* Ubicación
* Habitaciones
* Baños
* Parqueaderos
* Área
* Descripción

In [28]:
def scrape_page_data(driver, url):
    """Extrae el título (h1) y párrafos (p) de una página"""
    driver.get(url)
    time.sleep(2) # Espera a que la página cargue // Libreria Time 
    
    
## Tipo de propiedad
    try:
        h1 = driver.find_element(By.CSS_SELECTOR, 'h1[_ngcontent-detail-c331]').text
    except:
        h1 = "No encontrado"
    
    ## Precio
    try:
        price = driver.find_element(By.CSS_SELECTOR, 'span[_ngcontent-detail-c331][class="bold"]').text
    except:
        price = "No encontrado"

    ## Ubicación
    try:
        location_parts = [p.text for p in driver.find_elements(By.CSS_SELECTOR, "h3[_ngcontent-detail-c331][class='ng-star-inserted']") if p.text]
        location = ''.join(location_parts)
    except:
        location = "No encontrado"

    ## Habitaciones
    try:
        spaces = driver.find_elements(By.CSS_SELECTOR, 'div[_ngcontent-detail-c331][class="items ng-star-inserted"]')

        rooms = spaces[0].find_element(By.CSS_SELECTOR, 'p[_ngcontent-detail-c331][class="ng-star-inserted"]').text
    except:
        rooms = "No encontrado"
    
    ## Baños
    try:
        spaces = driver.find_elements(By.CSS_SELECTOR, 'div[_ngcontent-detail-c331][class="items ng-star-inserted"]')

        baths = spaces[1].find_element(By.CSS_SELECTOR, 'p[_ngcontent-detail-c331][class="ng-star-inserted"]').text
    except:
        baths = "No encontrado"

    try:
        spaces = driver.find_elements(By.CSS_SELECTOR, 'div[_ngcontent-detail-c331][class="items ng-star-inserted"]')

        ## Cuidado, no se contemplan casas con sólo habitaciones y garaje (sin baños)
        if len(spaces) > 1:
            garages = spaces[2].find_element(By.CSS_SELECTOR, 'p[_ngcontent-detail-c331][class="ng-star-inserted"]').text
        else:
            garages = 0
    except:
        garages = "No encontrado"


    ## Área
    try:

        features = driver.find_elements(By.XPATH, "//div[contains(@class, 'area-hasta ng-star-inserted')]")
        if len(features) > 1:
            area = features[1].find_element(By.XPATH, ".//span[2]").text
        else:
            area = features[0].find_element(By.XPATH, ".//span[2]").text
        
        area = area[:-3]
    except:
        area = "No encontrado"

    
    ## Descripción
    try:
        content = driver.find_element(By.CSS_SELECTOR, 'p[_ngcontent-detail-c328][class="expansion-panel__description"]').text
    except:
        content = "No encontrado"


    return {'url': url,
            'Tipo de propiedad': h1,
            'Precio': price,
            'Ubicación': location,
            'Habitaciones': rooms,
            'Baños': baths,
            'Parqueaderos': garages,
            'Área': area,
            'Descripción': content
            }

### 6. Funcion principal 💾

Me permite realizar el proceso repetitivo de recorrer cuántas páginas sea necesario y tomar el enlace de cada publicación. La información se guardará en formato CSV para un mejor análisis.

In [29]:
def main():
    # Número de páginas a recorrer
    num_pages = 1  # Puedes ajustar este número según cuántas páginas quieras procesar

    # URL base sin el parámetro de página
    url_base = "https://www.ciencuadras.com/arriendo/medellin/apartamento?q=medellin"

    # Inicializar driver
    driver = setup_driver()
    try:
        all_links = []
        # Iterar sobre cada página
        for page in range(1, num_pages + 1):
            url = f"{url_base}{page}"
            print(f"Obteniendo enlaces de la página {page}: {url}")
            page_links = get_links(driver, url)
            print(f"Se encontraron {len(page_links)} enlaces en la página {page}")
            all_links.extend(page_links)

        print(f"Total de enlaces obtenidos: {len(all_links)}")
        
        if not all_links:
            print("No se encontraron enlaces en ninguna página. Revisa los selectores o la URL.")
            return
        

# Paso 2: Visitar cada enlace y extraer información

        datos = []
        print("Iniciando extracción de datos...")
        for i, enlace in enumerate(all_links, 1):
            print(f"Procesando enlace {i}/{len(all_links)}: {enlace}")
            try:
                datos_pagina = scrape_page_data(driver, enlace)
                datos.append(datos_pagina)
            except Exception as e:
                print(f"Error al procesar {enlace}: {str(e)}")
                continue
        

# Paso 3: Crear DataFrame con los datos

        df = pd.DataFrame(datos)  #Aqui se utilizará la libreria pandas para guardar los datos en CSV.
        

        # Mostrar resultados
        print("\nResumen de datos obtenidos:")
        print(df.head())
        
        # Guardar a CSV (opcional)
        df.to_csv('datos_extraidos.csv', index=False)
        print("\nDatos guardados en 'datos_extraidos.csv'")
    
    finally:
        # Cerrar el driver siempre
        driver.quit()
        print("Driver cerrado.")

if __name__ == "__main__":
    main()

Obteniendo enlaces de la página 1: https://www.ciencuadras.com/arriendo/medellin/apartamento?q=medellin1
Se encontraron 20 enlaces en la página 1
Total de enlaces obtenidos: 20
Iniciando extracción de datos...
Procesando enlace 1/20: https://www.ciencuadras.com/inmueble/apartamento-en-arriendo-en-villanueva-medellin-3306914
Procesando enlace 2/20: https://www.ciencuadras.com/inmueble/apartamento-en-arriendo-en-laureles-medellin-3314461
Procesando enlace 3/20: https://www.ciencuadras.com/inmueble/apartamento-en-arriendo-en-simesa-medellin-3296837
Procesando enlace 4/20: https://www.ciencuadras.com/inmueble/apartamento-en-arriendo-en-caicedo-medellin-3314487
Procesando enlace 5/20: https://www.ciencuadras.com/inmueble/apartamento-en-arriendo-en-el-tesoro-medellin-3307753
Procesando enlace 6/20: https://www.ciencuadras.com/inmueble/apartamento-en-arriendo-en-belen-medellin-3276974
Procesando enlace 7/20: https://www.ciencuadras.com/inmueble/apartamento-en-arriendo-en-naranjitos-medellin-3